# Translate Manifests to Ukrainian

This notebook fills `target_text_uk` in the JSONL manifests created for the UA AST task.
It is resumable: translated output is written line by line, so rerunning continues from existing files.


In [9]:
from pathlib import Path
import json
import time
from typing import Iterable

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from tqdm.auto import tqdm


In [10]:
INPUT_DIR = Path('ua_ast_data/manifests/combined')
OUTPUT_DIR = Path('ua_ast_data/manifests/combined_uk')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = [
    'train',
    'validation.clean',
    'validation.other',
    'test.clean',
    'test.other',
]

# Good open multilingual MT model for English -> Ukrainian.
# First run may download the model from Hugging Face.
MODEL_NAME = 'facebook/nllb-200-distilled-600M'
SRC_LANG = 'eng_Latn'
TGT_LANG = 'ukr_Cyrl'

BATCH_SIZE = 8
MAX_INPUT_LENGTH = 256
MAX_NEW_TOKENS = 256


In [11]:
def pick_device() -> str:
    if torch.cuda.is_available():
        return 'cuda'
    if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'

DEVICE = pick_device()
print('Device:', DEVICE)


Device: mps


In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang=SRC_LANG)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

forced_bos_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)
print('Loaded:', MODEL_NAME)


Loading weights: 100%|██████████| 512/512 [00:00<00:00, 67601.08it/s]


Loaded: facebook/nllb-200-distilled-600M


In [13]:
def read_jsonl(path: Path) -> Iterable[dict]:
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                yield json.loads(line)


def count_lines(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open('r', encoding='utf-8') as f:
        return sum(1 for _ in f)


def batched(items, batch_size: int):
    batch = []
    for item in items:
        batch.append(item)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch


In [14]:
@torch.inference_mode()
def translate_batch(texts: list[str]) -> list[str]:
    encoded = tokenizer(
        texts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
    )
    encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
    generated = model.generate(
        **encoded,
        forced_bos_token_id=forced_bos_token_id,
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=4,
    )
    return tokenizer.batch_decode(generated, skip_special_tokens=True)


In [15]:
def translate_manifest(split_name: str) -> None:
    in_path = INPUT_DIR / f'{split_name}.jsonl'
    out_path = OUTPUT_DIR / f'{split_name}.jsonl'

    if not in_path.exists():
        print(f'Skip missing split: {in_path}')
        return

    already_done = count_lines(out_path)
    total = count_lines(in_path)
    remaining = max(total - already_done, 0)
    print(f'\n{split_name}: {already_done}/{total} already translated')

    if remaining == 0:
        print(f'{split_name}: done')
        return

    records_iter = read_jsonl(in_path)
    for _ in range(already_done):
        next(records_iter, None)

    progress = tqdm(
        total=total,
        initial=already_done,
        desc=split_name,
        unit='sample',
        dynamic_ncols=True,
    )

    with out_path.open('a', encoding='utf-8') as out_f:
        try:
            for batch_records in batched(records_iter, BATCH_SIZE):
                texts = [record['source_text'] for record in batch_records]
                translations = translate_batch(texts)

                for record, translation in zip(batch_records, translations):
                    record['target_text_uk'] = translation.strip()
                    out_f.write(json.dumps(record, ensure_ascii=False) + '\n')

                out_f.flush()
                progress.update(len(batch_records))
        finally:
            progress.close()


In [16]:
# Start with validation/test to verify quality quickly.
# Then run train; it is much larger.
for split in ['validation.clean', 'validation.other', 'test.clean', 'test.other']:
    translate_manifest(split)



validation.clean: 136/2703 already translated


validation.clean: 100%|██████████| 2703/2703 [56:24<00:00,  1.32s/sample]



validation.other: 0/2864 already translated


validation.other: 100%|██████████| 2864/2864 [56:52<00:00,  1.19s/sample]



test.clean: 0/2620 already translated


test.clean: 100%|██████████| 2620/2620 [55:50<00:00,  1.28s/sample]



test.other: 0/2939 already translated


test.other: 100%|██████████| 2939/2939 [1:05:50<00:00,  1.34s/sample]


In [17]:
# Run this after validation/test look good.
translate_manifest('train')



train: 0/28539 already translated


train: 100%|██████████| 28539/28539 [13:30:29<00:00,  1.70s/sample]


In [18]:
# Quick quality/sample check.
sample_path = OUTPUT_DIR / 'validation.clean.jsonl'
with sample_path.open('r', encoding='utf-8') as f:
    for _, line in zip(range(5), f):
        record = json.loads(line)
        print('EN:', record['source_text'])
        print('UK:', record['target_text_uk'])
        print()


EN: the long drizzle had begun pedestrians had turned up collars and trousers at the bottom
UK: довгий дождж почав пішоходів з'явилися ожерельки і штани внизу

EN: many little wrinkles gathered between his eyes as he contemplated this and his brow moistened
UK: Багато малих морщин зібралися між його очима, коли він розмірковував це і його лба ввлажня

EN: then he rang the bell no answer
UK: Тоді він звонів на дзвінку, ні на що не відповів.

EN: he could arrange that satisfactorily for carrie would be glad to wait if necessary
UK: Він міг організувати, що задовольняє для Керрі був би радий чекати, якщо це необхідно.

EN: his first impulse was to write but four words in reply go to the devil
UK: Його першим імпульсом було написати, але чотири слова в відповідь йдуть до д'явола.

